# 06쪽 · Reducer: 덮어쓰기와 이어 붙이기

## 준비
아래 셀은 파일 경로와 `.env` 로더를 준비합니다. 한 번 실행하고 넘어가세요.

In [ ]:
from pathlib import Path
import sys

# 노트북을 연 위치가 ipynb 폴더이면 한 단계 위로 이동합니다.
ROOT = Path.cwd()

if ROOT.name == "ipynb":
    ROOT = ROOT.parent

if not (ROOT / "lab" / "config.py").exists():
    raise RuntimeError("Langgraph_6_Week_Code 또는 ipynb 폴더에서 실행하세요.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lab.config import make_llm, load_settings

## A-1. Reducer 없는 State

먼저 기본 동작만 끝까지 실행합니다.

In [ ]:
from typing_extensions import TypedDict, NotRequired
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    a: int
    b: int
    c: NotRequired[int]
    d: NotRequired[int]
    events: list[str]

## A-2. 첫 번째와 두 번째 기록

PPT의 node1, node2입니다.

In [ ]:
def node1(state):
    return {
        "c": state["a"] + state["b"],
        "events": ["첫 번째"],
    }


def node2(state):
    return {
        "d": state["c"] * 2,
        "events": ["두 번째"],
    }

## A-3. 그래프 만들기

node1 다음 node2를 실행합니다.

In [ ]:
builder = StateGraph(State)

builder.add_node("node1", node1)
builder.add_node("node2", node2)

builder.add_edge(START, "node1")
builder.add_edge("node1", "node2")
builder.add_edge("node2", END)

graph = builder.compile()

## A-4. 기본 동작 결과

최종 events에 무엇이 남을까요?

In [ ]:
result = graph.invoke({
    "a": 10,
    "b": 20,
    "events": [],
})

print(result["events"])

**결과 해설**

`["두 번째"]`만 남습니다. 같은 키의 새 값이 이전 값을 교체합니다.

## B-1. operator.add를 붙인 State

여기서부터 두 번째 예제입니다. 달라지는 부분은 events의 타입 한 줄입니다. 아래에서도 그래프를 새로 만듭니다.

In [ ]:
from typing import Annotated
import operator


class State(TypedDict):
    a: int
    b: int
    c: NotRequired[int]
    d: NotRequired[int]
    events: Annotated[list[str], operator.add]

## B-2. 같은 Node 정의

A와 같은 계산입니다. 두 예제를 쉽게 비교하도록 다시 적었습니다.

In [ ]:
def node1(state):
    return {
        "c": state["a"] + state["b"],
        "events": ["첫 번째"],
    }


def node2(state):
    return {
        "d": state["c"] * 2,
        "events": ["두 번째"],
    }

## B-3. 그래프 새로 만들기

B의 State를 사용하는 그래프입니다.

In [ ]:
builder = StateGraph(State)

builder.add_node("node1", node1)
builder.add_node("node2", node2)

builder.add_edge(START, "node1")
builder.add_edge("node1", "node2")
builder.add_edge("node2", END)

graph = builder.compile()

## B-4. operator.add 결과

A와 같은 입력으로 실행합니다.

In [ ]:
result = graph.invoke({
    "a": 10,
    "b": 20,
    "events": [],
})

print(result["events"])

**결과 해설**

`["첫 번째", "두 번째"]`가 남습니다. 두 리스트를 이어 붙입니다.

## 비교

| 예제 | 최종 events |
|---|---|
| A: 기본 동작 | `["두 번째"]` |
| B: operator.add | `["첫 번째", "두 번째"]` |